# Global Solution 2026.2 — Generative AI and Advanced Nets (FIAP)
## Projeto: Radar Cidadão
> *"Radar Cidadão: cidadãos registram problemas da cidade, sensores confirmam os riscos e a população recebe alertas para se proteger de alagamentos."*

Repositório: https://github.com/m4ntonio/radar-cidadao

---

### Identificação do Grupo
| Integrantes | RM |
| :--- | :--- |
| Mario Antonio Oliveira | 567713 |
| Sarah Marangoni Furlan | 566764 |
| Vinicius Augusto Silva | 566830 |

---

## Etapa 1: Definição do Problema, Público, Escopo e Integração CitizenTech

### 1.1. Problema CitizenTech Escolhido
Alagamentos em São Paulo causam perdas materiais, interrompem a circulação viária e colocam vidas humanas em risco. Parte substancial do problema decorre de bueiros e bocas de lobo entupidos por lixo domiciliar e entulho descartados em via pública. A informação sobre esses riscos chega tarde tanto à população quanto ao poder público, levando cidadãos a adotarem condutas de risco durante tempestades (como tentar atravessar enxurradas a pé ou de veículo).

### 1.2. Público Impactado e Contexto de Aplicação
- **Público Primário:** Moradores, pedestres e motoristas que circulam por áreas sujeitas a alagamento em São Paulo (bacias hidrográficas do Tamanduateí, Aricanduva, Anhangabaú, Pirajuçara, Tietê e Pinheiros).
- **Destino dos Encaminhamentos (Poder Público):** As 32 Subprefeituras do Município de São Paulo (zeladoria e desobstrução de microdrenagem) e a **Defesa Civil** (vistoria preventiva, alertas e assistência).

### 1.3. Escopo do MVP e Alinhamento com os ODS (ONU)
- **Escopo do MVP:** Plataforma de zeladoria urbana colaborativa com foco principal em **chuva, bueiros obstruídos e alagamento**. Outros problemas urbanos (buracos no asfalto, iluminação pública e queda de árvore) ficam mapeados como expansão futura e não fazem parte do MVP.
- **Complementaridade Oficial:** O **Radar Cidadão** complementa os canais oficiais e não os substitui. Em situação de emergência, a orientação obrigatória é ligar **199 (Defesa Civil)** ou **193 (Corpo de Bombeiros)**. Para zeladoria preventiva em tempo seco, utiliza-se a Central **156** e o **encaminhamento** às subprefeituras.
- **Objetivos de Desenvolvimento Sustentável (ODS):**
  - **Principal — ODS 11 (Cidades e Comunidades Sustentáveis), Meta 11.5:** Reduzir mortes e perdas causadas por desastres, incluindo os relacionados à água.
  - **Secundário — ODS 13 (Ação Climática), Meta 13.1:** Reforçar a resiliência e a capacidade de adaptação a riscos relacionados ao clima.

### 1.4. Integração no Fluxo Único de 8 Etapas do Radar Cidadão
| Etapa | Componente do Fluxo | Disciplina Responsável | Natureza no MVP |
| :---: | :--- | :--- | :---: |
| **1** | **Relato:** O cidadão registra o problema no app com foto, localização e categoria. | Front End & Mobile (React + Vite) | Real |
| **2** | **Análise de Imagem:** Detector identifica lixo ou entulho em via pública e informa a confiança. | Visão Computacional | Real |
| **3** | **Confirmação por Sensores:** Sensores de nível de água e de chuva confirmam o risco na região. | IoT (ESP32 / MQTT) | Simulado |
| **4** | **Memória de Alagamento:** Circuito neuromórfico registra que houve alagamento no ponto após a água baixar. | Computação Neuromórfica (Falstad) | Simulado |
| **5** | **Mapa de Risco:** O relato validado atualiza o Mapa de Risco da região. | Front End & Mobile | Real |
| **6** | **Orientação (ESTE NOTEBOOK):** A IA responde ao cidadão com base nos documentos oficiais da Defesa Civil. | **IA Generativa + Fábrica de Corpus (NLP)** | **Real** |
| **7** | **Alerta:** A população da região recebe o alerta preventivo ou de emergência. | IoT + Front End & Mobile | Real / Simulado |
| **8** | **Encaminhamento:** O robô consolida relatos e dados de chuva e gera o encaminhamento para o poder público. | RPA + IA | Real (gerado) |


## Etapa 2: Escolha e Justificativa da Abordagem de Inteligência Artificial

### 2.1. Abordagens Selecionadas da Disciplina
Para atender ao requisito de que a solução **não pode ser um chatbot genérico**, esta aplicação combina três abordagens técnicas estudadas na disciplina:
1. **Transformers e Foundation Models:**
   - Modelo de Embeddings Multilíngue (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`) para representação vetorial semântica em língua portuguesa (384 dimensões).
   - Modelo Generativo de Linguagem via infraestrutura **NVIDIA NIM** (`build.nvidia.com`), utilizando o modelo de alta velocidade e precisão (`nvidia/nemotron-3.5-bolt-e4b`), operando com streaming Server-Sent Events (`text/event-stream; charset=utf-8`) e temperatura determinística (`0.1`).
2. **RAG (Retrieval-Augmented Generation) com Integração Interdisciplinar:**
   - Indexação vetorial no **ChromaDB** do corpus estruturado fornecido pela disciplina de **Fábrica de Corpus (NLP)** (`corpus_rag.jsonl`), composto por **9 documentos oficiais** (Ministério das Cidades, SEDEC e Defesa Civil) pré-fatiados em **355 pedaços (chunks)** com rastreamento de autoria e páginas.
   - Camada de contingência (**Fallback**): suporte automático aos 4 manuais técnicos municipais/estaduais em `.txt` (22 páginas) caso a fonte primária esteja inacessível.
   - Recuperação semântica dos trechos e citações oficiais pertinentes ao **relato** do cidadão antes da geração da resposta.
3. **Prompt Engineering e Arquitetura de Guardrails (Roteamento de Segurança):**
   - **Guardrail de Emergência (`193` / `199`):** Camada determinística que identifica indicadores de risco letal imediato (pessoas ilhadas, veículo arrastado, correnteza severa, desabamento) e injeta o protocolo de socorro no topo da resposta.
   - **Guardrail de Contenção de Escopo (Anti-Alucinação):** Camada estrita que bloqueia na entrada perguntas alheias ao domínio hidrológico e da Defesa Civil (como perguntas pessoais, culinária, esportes ou problemas fora do MVP), impedindo que o modelo opere como chatbot genérico.

### 2.2. Justificativa Técnica Comparativa
| Critério Técnico | Chatbot Genérico (LLM Zero-Shot Sem RAG) | Fine-Tuning Estático | RAG + Guardrails (Abordagem Adotada no Radar Cidadão) |
| :--- | :--- | :--- | :--- |
| **Ancoragem em Normas Locais e Federais** | Baixa: desconhece procedimentos específicos de subprefeituras e estados do CGE-SP. | Média: exige re-treinamento a cada revisão de protocolo municipal. | **Alta:** consulta diretamente os 9 documentos oficiais do corpus de NLP indexados no ChromaDB com fallback local. |
| **Risco de Alucinação em Crise** | Alto: pode sugerir atravessar áreas alagadas com "cuidado" ou desentupir bueiro na chuva. | Médio: não garante citação literal nem bloqueio determinístico. | **Mitigado:** restrição estrita ao contexto recuperado + bloqueio determinístico de condutas perigosas. |
| **Rastreabilidade e Auditoria** | Inexistente: não informa de qual documento ou página tirou a informação. | Inexistente: pesos neurais opacos sem citação documental. | **Verificável:** retorna a fonte oficial, o título da cartilha e as páginas exatas do documento consultado. |


## Etapa 3: Implementação Técnica — Instalação, Credenciais, Integração com Fábrica de Corpus (NLP) e ChromaDB
### 3.1. Instalação de Pacotes e Dependências


In [ ]:
# Instalação dos pacotes necessários para o pipeline RAG, ChromaDB e interface Gradio
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

pacotes = [
    "langchain",
    "langchain-community",
    "langchain-core",
    "langchain-text-splitters",
    "chromadb",
    "sentence-transformers",
    "gradio",
    "pypdf",
    "requests",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", *pacotes],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Status: pacotes instalados no ambiente.")

Status: pacotes instalados no ambiente.


### 3.2. Configuração Segura de Credenciais da API (NVIDIA NIM)
Carregamento da chave `NVIDIA_API_KEY` via **Google Colab Secrets** (`userdata.get('NVIDIA_API_KEY')`) ou entrada protegida via `getpass`, mantendo as credenciais fora do código-fonte.


In [16]:
import os
import getpass

try:
    from google.colab import userdata
    nvidia_api_key = userdata.get('NVIDIA_API_KEY')

    if not nvidia_api_key:
        raise ValueError("Chave não localizada nos secrets do Colab.")

except Exception:
    nvidia_api_key = os.environ.get("NVIDIA_API_KEY")

    if not nvidia_api_key:
        print("\nAcesse o NVIDIA Build para obter sua API Key:")
        print("https://build.nvidia.com/?modal=signin\n")
        nvidia_api_key = getpass.getpass("Digite sua NVIDIA_API_KEY: ")

nvidia_api_key = nvidia_api_key.strip()
os.environ["NVIDIA_API_KEY"] = nvidia_api_key

print(f"Credencial NVIDIA NIM configurada: {nvidia_api_key[:6]}...{nvidia_api_key[-4:]}")


Credencial NVIDIA NIM configurada: nvapi-...4eFD


### 3.3. Carga do Corpus da Fábrica de Corpus (NLP) com Arquitetura de Contingência
Atendendo à integração interdisciplinar entre a **Fábrica de Corpus (NLP)** e a **Etapa 6 (IA Generativa)**, a carga dos dados opera sob uma arquitetura de alta disponibilidade em 3 níveis:

1. **Nível 1 (Fonte Primária — Fábrica de Corpus NLP):** Consome a esteira pré-processada pela integrante **Sarah Marangoni Furlan** (`corpus_rag.jsonl`), contendo **9 documentos governamentais oficiais** (Ministério das Cidades, SEDEC e Defesa Civil), estruturados em **355 pedaços (chunks)** com metadados de autoria e páginas.
2. **Nível 2 (Fallback Remoto — Monorepo GitHub):** Caso a URL de NLP esteja indisponível, aciona a API do GitHub para baixar pontualmente os arquivos técnicos `.txt` da pasta `GENAI/corpus` do repositório (`m4ntonio/radar-cidadao`), totalizando 22 páginas demarcadas.
3. **Nível 3 (Fallback Local — Armazenamento em Disco):** Utiliza arquivos `.txt` já existentes na pasta `./corpus` do ambiente, garantindo que o pipeline nunca fique desprovido de base documental.


In [19]:
import os
import re
import json
import requests
from pathlib import Path

# ---------------------------------------------------------------
# CONFIGURAÇÃO DAS FONTES
# ---------------------------------------------------------------
# Fonte primária: saída da disciplina Fábrica de Corpus (NLP)
URL_CORPUS_NLP = (
    "https://raw.githubusercontent.com/sarahmfurlan-sudo/"
    "radar-cidadao/main/nlp-fabrica-corpus/dados/saida_rag/corpus_rag.jsonl"
)

# Fonte de contingência: pasta GENAI/corpus do monorepo do grupo (via API do GitHub)
repo_owner = "m4ntonio"
repo_name = "radar-cidadao"
branch = "main"
api_url = (
    f"https://api.github.com/repos/{repo_owner}/"
    f"{repo_name}/contents/GENAI/corpus?ref={branch}"
)


def citacao(p):
    paginas = p.get("paginas") or []
    if not paginas:
        return p.get("titulo", "Documento Oficial")
    pags = f"p. {paginas[0]}" if len(paginas) == 1 else f"p. {paginas[0]}-{paginas[-1]}"
    return f"{p.get('titulo', 'Documento Oficial')}, {pags}"


def validar_pedacos(linhas):
    """Converte linhas JSONL em dicionários e descarta registros sem texto."""
    pedacos = []
    for l in linhas:
        if not l.strip():
            continue
        try:
            p = json.loads(l)
        except json.JSONDecodeError:
            continue
        if isinstance(p, dict) and str(p.get("texto", "")).strip():
            pedacos.append(p)
    return pedacos


def baixar_corpus_txt_github(destino: Path) -> int:
    """Baixa apenas os arquivos de GENAI/corpus pela API do GitHub. Retorna a quantidade baixada."""
    destino.mkdir(parents=True, exist_ok=True)
    print(f"Consultando corpus oficial no monorepo:\n{api_url}\n")

    response = requests.get(api_url, timeout=30)
    response.raise_for_status()

    arquivos = sorted(
        [
            item for item in response.json()
            if item["type"] == "file" and item["name"].lower().endswith(".txt")
        ],
        key=lambda item: item["name"].lower(),
    )
    if not arquivos:
        raise RuntimeError("Nenhum arquivo encontrado em GENAI/corpus.")

    print(f"Arquivos encontrados: {len(arquivos)}")
    for arquivo in arquivos:
        print(f" - Baixando: {arquivo['name']}")
        download = requests.get(arquivo["download_url"], timeout=60)
        download.raise_for_status()
        (destino / arquivo["name"]).write_bytes(download.content)

    return len(arquivos)


# Variáveis consumidas pela Célula 3.4
pedacos_corpus = []
modo_corpus = "NLP_FABRICA"
corpus_path = Path("./corpus")

print("=" * 92)
print("INTEGRAÇÃO INTERDISCIPLINAR: FÁBRICA DE CORPUS (NLP) -> IA GENERATIVA")
print("=" * 92)

# NÍVEL 1: corpus JSONL da Fábrica de Corpus (NLP)
erro_nivel_1 = None

try:
    print(f"[Nível 1] Baixando corpus da Fábrica de NLP:\n{URL_CORPUS_NLP}\n")
    resp = requests.get(URL_CORPUS_NLP, timeout=20)
    resp.raise_for_status()
    resp.encoding = "utf-8"

    pedacos_corpus = validar_pedacos(resp.text.splitlines())
    if not pedacos_corpus:
        raise ValueError("arquivo JSONL vazio ou sem campo 'texto'")

    print("Status: corpus da Fábrica de NLP carregado.")

except Exception as erro:
    erro_nivel_1 = type(erro).__name__
    print(f"[ERROR] [Nível 1] indisponível: {type(erro).__name__}")
    pedacos_corpus = []
    modo_corpus = "FALLBACK_LOCAL"

    # NÍVEL 2: arquivos da pasta GENAI/corpus via API do GitHub
    try:
        print("\n[Nível 2] Acionando FALLBACK: download dos arquivos do monorepo...")
        baixar_corpus_txt_github(corpus_path)
    except Exception as erro_api:
        print(f"[ERROR] [Nível 2] indisponível: {type(erro_api).__name__}\n")

        # NÍVEL 3: arquivos já presentes em ./corpus (upload manual ou execução anterior)
        if any(corpus_path.glob("*.txt")):
            print("[Nível 3] Usando arquivos já existentes em ./corpus.")
        else:
            print("[ERROR] [Nível 3] indisponível: pasta ./corpus vazia.\n")

            class CorpusIndisponivel(Exception):
                """Interrompe a célula sem exibir traceback no Colab/Jupyter."""
                def _render_traceback_(self):
                    return [
                        "=" * 92,
                        "⚠︎ NENHUMA FONTE DE CORPUS DISPONÍVEL — EXECUÇÃO INTERROMPIDA",
                        "=" * 92,
                        "Verifique:",
                        f"1. URL da Fábrica de NLP:\n{URL_CORPUS_NLP}",
                        f"2. Repositório do grupo:  https://github.com/{repo_owner}/{repo_name}",
                        "3. Ou faça upload manual dos arquivos para a pasta ./corpus no Colab.",
                        "=" * 92,
                    ]

            raise CorpusIndisponivel() from None

# VERIFICAÇÃO DA FONTE ATIVA
print("=" * 92)
print("VERIFICAÇÃO DO CORPUS OFICIAL DA DEFESA CIVIL (REQUISITO: 3 A 5 DOCS, >= 20 PÁGINAS)")
print("=" * 92)
print(f"Modo do corpus ativo: {modo_corpus}")

if modo_corpus == "NLP_FABRICA":
    print(f" - Total de pedaços (chunks pré-fatiados): {len(pedacos_corpus)}")
    print(f" - Documentos oficiais únicos: {len({p.get('fonte', '') for p in pedacos_corpus})}")
    print(f" - Exemplo de citação estruturada: {citacao(pedacos_corpus[0])}")
else:
    total_bytes = 0
    total_paginas = 0
    arquivos_txt = sorted(corpus_path.glob("*.txt"))
    for f in arquivos_txt:
        texto = f.read_text(encoding="utf-8")
        size = f.stat().st_size
        paginas_doc = len(re.findall(r"\[PÁGINA \d+ de \d+\]", texto))
        total_bytes += size
        total_paginas += paginas_doc
        print(f" - Arquivo: {f.name:<42} | Páginas: {paginas_doc:>2} | Tamanho: {size:>6} bytes")
    print("-" * 92)
    print(f"Total de Documentos: {len(arquivos_txt)} documentos oficiais (CEDEC-SP, COMDEC-SP, SEDEC/MDR, CGE-SP)")
    print(f"Total de Páginas Verificadas: {total_paginas} páginas (Requisito mínimo: 20 páginas)")
    print(f"Volume Total de Texto: {total_bytes} bytes")

print("=" * 92)

INTEGRAÇÃO INTERDISCIPLINAR: FÁBRICA DE CORPUS (NLP) -> IA GENERATIVA
[Nível 1] Baixando corpus da Fábrica de NLP:
https://raw.githubusercontent.com/sarahmfurlan-sudo/radar-cidadao/main/nlp-fabrica-corpus/dados/saida_rag/corpus_rag.jsonl

Status: corpus da Fábrica de NLP carregado.
VERIFICAÇÃO DO CORPUS OFICIAL DA DEFESA CIVIL (REQUISITO: 3 A 5 DOCS, >= 20 PÁGINAS)
Modo do corpus ativo: NLP_FABRICA
 - Total de pedaços (chunks pré-fatiados): 355
 - Documentos oficiais únicos: 9
 - Exemplo de citação estruturada: Periferia Sem Risco, do Ministério das Cidades (2024), p. 3-5


### 3.4. Estruturação Semântica e Indexação Vetorial no ChromaDB
- **Fluxo Primário (Fábrica de NLP):** Como os 355 pedaços já chegam pré-fatiados e enriquecidos pela esteira de NLP com metadados estruturados (`titulo`, `fonte`, `paginas` e função de `citacao`), os registros são convertidos diretamente em objetos `Document` da LangChain sem necessidade de novo fatiamento textual.
- **Fluxo de Contingência (Fallback Local):** Se ativado o modo de arquivos `.txt`, os documentos são processados e fatiados pelo `RecursiveCharacterTextSplitter` (tamanho de 650 caracteres e sobreposição de 90 caracteres).
- **Vetorização e Armazenamento:** Os documentos são incorporados pelo modelo `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384 dimensões, execução local em CPU) e indexados na coleção vetorial persistente do **ChromaDB** com busca top-k ($k=2$).


In [ ]:
import os
import re
import warnings
warnings.filterwarnings("ignore")
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

split_docs = []

# Caso A: Corpus estruturado da Fábrica de NLP (chunks já vêm prontos e limpos)
if modo_corpus == "NLP_FABRICA" and pedacos_corpus:
    print(f"Carregando {len(pedacos_corpus)} pedaços pré-fatiados pela esteira de NLP...")
    for p in pedacos_corpus:
        pags = p.get("paginas") or []
        pag_str = f"p. {pags[0]}" if len(pags) == 1 else (f"p. {pags[0]}-{pags[-1]}" if pags else "p. única")
        doc = Document(
            page_content=p["texto"],
            metadata={
                "source_file": p.get("fonte", "Defesa Civil"),
                "titulo": p.get("titulo", "Documento Oficial"),
                "pagina": pag_str,
                "citacao": citacao(p)
            }
        )
        split_docs.append(doc)
    print(f"Total de documentos estruturados para indexação: {len(split_docs)}")

# Caso B: Fallback local para arquivos .txt
else:
    print("Processando arquivos .txt locais via RecursiveCharacterTextSplitter...")
    raw_page_docs = []
    for txt_file in sorted(corpus_path.glob("*.txt")):
        content = txt_file.read_text(encoding="utf-8")
        partes = re.split(r"(\[PÁGINA \d+ de \d+\])", content)
        cabecalho = partes[0].strip()
        pagina_atual = "Página 1"
        for item in partes[1:]:
            if re.match(r"\[PÁGINA \d+ de \d+\]", item):
                pagina_atual = item.strip("[]")
            else:
                texto_pagina = f"{cabecalho.splitlines()[0]}\n[{pagina_atual}]\n{item.strip()}"
                raw_page_docs.append(Document(
                    page_content=texto_pagina,
                    metadata={"source_file": txt_file.name, "pagina": pagina_atual, "citacao": f"{txt_file.name} ({pagina_atual})"}
                ))
    splitter = RecursiveCharacterTextSplitter(chunk_size=650, chunk_overlap=90)
    split_docs = splitter.split_documents(raw_page_docs)

# Vetorização e Indexação no ChromaDB
print("Inicializando modelo de embeddings multilíngue (paraphrase-multilingual-MiniLM-L12-v2)...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings_model,
    collection_name="radar_cidadao_defesa_civil_v3"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Base vetorial ChromaDB operacional (busca top-k = 2 chunks com citações oficiais).")

Processando arquivos .txt locais via RecursiveCharacterTextSplitter...
Inicializando modelo de embeddings multilíngue (paraphrase-multilingual-MiniLM-L12-v2)...


/tmp/ipykernel_1974/2959522442.py:56: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Base vetorial ChromaDB operacional (busca top-k = 2 chunks com citações oficiais).


## Etapa 4: Construção e Integração do Fluxo da Solução (RAG + Guardrails + Fluxo Único)

O pipeline abaixo implementa a **Etapa 6 (Orientação)** integrada às demais etapas do **Radar Cidadão**:
- **Entradas integradas:** Recebe o **relato** do cidadão (Etapa 1), a confirmação da análise de imagem de lixo/entulho (Etapa 2), os dados simulados dos sensores de chuva e nível de água (Etapa 3), o status da memória de alagamento (Etapa 4) e a localização no **Mapa de Risco** (Etapa 5).
- **Processamento IA Generativa:** Passa pelos guardrails de contenção de escopo e de emergência (`193` / `199`), recupera os trechos e páginas do corpus da **Defesa Civil** no ChromaDB e gera a orientação via Streaming SSE no **NVIDIA NIM**.
- **Saídas integradas:** Entrega a orientação fundamentada ao cidadão (Etapa 6) e gera a estrutura JSON para disparo do **alerta** (Etapa 7) e consolidação do **encaminhamento** ao poder público (Etapa 8).


In [ ]:
import os
import re
import json
import time
import requests
from langchain_core.prompts import PromptTemplate

INVOKE_URL = "https://integrate.api.nvidia.com/v1/chat/completions"

# 1. Descoberta automática do endpoint ativo na conta NVIDIA NIM
def selecionar_endpoint_ativo(api_key: str):
  candidatos = [
      ("nvidia/nemotron-3.5-bolt-e4b", "text"),  # ⚡ Campeão: 0.95s!
      ("minimaxai/minimax-m3.1-flash", "text"),
      ("z-ai/glm-5.3-flash", "text"),
      ("meta/llama-3.2-90b-vision-instruct", "vision_list"),  # Fallback
  ]
  headers = {
      "Authorization": f"Bearer {api_key}",
      "Accept": "application/json",
      "Content-Type": "application/json",
  }
  for model_name, fmt in candidatos:
    try:
      content_payload = (
          [{"type": "text", "text": "OK"}] if fmt == "vision_list" else "OK"
      )
      payload = {
          "model": model_name,
          "messages": [{"role": "user", "content": content_payload}],
          "max_tokens": 10,
          "temperature": 0.1,
          "stream": False,
      }
      t0 = time.time()
      r = requests.post(INVOKE_URL, headers=headers, json=payload, timeout=10)
      if r.status_code == 200:
        dt = time.time() - t0
        print(
            f"Endpoint NVIDIA NIM selecionado: {model_name} (Status HTTP 200 em"
            f" {dt:.2f}s)"
        )
        return model_name, fmt
    except Exception:
      continue
  return "nvidia/nemotron-3.5-bolt-e4b", "text"

MODELO_ATIVO, FORMATO_PAYLOAD = selecionar_endpoint_ativo(nvidia_api_key)

# 2. Chamada ao LLM via Streaming SSE com codificação estrita UTF-8
def chamar_nvidia_nim(prompt_input, max_tokens: int = 260) -> str:
  prompt_text = (
      prompt_input.to_string()
      if hasattr(prompt_input, "to_string")
      else str(prompt_input)
  )
  content_payload = (
      [{"type": "text", "text": prompt_text}]
      if FORMATO_PAYLOAD == "vision_list"
      else prompt_text
  )

  headers_stream = {
      "Authorization": f"Bearer {nvidia_api_key}",
      "Accept": "text/event-stream",
      "Content-Type": "application/json; charset=utf-8",
  }
  payload_stream = {
      "model": MODELO_ATIVO,
      "messages": [{"role": "user", "content": content_payload}],
      "max_tokens": max_tokens,
      "temperature": 0.1,
      "top_p": 0.9,
      "stream": True,
  }

  with requests.post(
      INVOKE_URL,
      headers=headers_stream,
      json=payload_stream,
      stream=True,
      timeout=(10, 45),
  ) as resp:
    resp.raise_for_status()
    # Força explicitamente o encoding UTF-8 na resposta
    resp.encoding = "utf-8"

    ctype = resp.headers.get("Content-Type", "")
    if "text/event-stream" in ctype:
      partes = []
      # Usa decode_unicode=False para receber bytes puros e decodificar estritamente em UTF-8
      for raw_line in resp.iter_lines(decode_unicode=False):
        if not raw_line:
          continue
        try:
          line = raw_line.decode("utf-8")
        except UnicodeDecodeError:
          line = raw_line.decode("utf-8", errors="replace")

        if line.startswith("data:"):
          data_str = line[5:].strip()
          if data_str == "[DONE]":
            break
          try:
            chunk_json = json.loads(data_str)
            delta = chunk_json["choices"][0].get("delta", {})
            token = delta.get("content", "")
            if token:
              partes.append(token)
          except Exception:
            continue

      texto_final = "".join(partes).strip()
    else:
      data = resp.json()
      texto_final = data["choices"][0]["message"]["content"].strip()

  if not texto_final:
    raise RuntimeError("Resposta vazia do endpoint de streaming.")

  # Sanitização defensiva contra mojibake (se ainda vier Latin-1 duplicado)
  if "Ã£" in texto_final or "Ã©" in texto_final or "Ã¡" in texto_final:
    try:
      texto_final = texto_final.encode("latin-1").decode("utf-8")
    except Exception:
      pass

  return texto_final

# 3. Guardrail 1: Detecção de Emergência Imediata (193 Bombeiros / 199 Defesa Civil)
def check_emergency_priority(text: str) -> bool:
    emergency_keywords = [
        r"socorro", r"ilhados?", r"afogand", r"carro sendo arrastado",
        r"arrastando meu carro", r"arrastou o carro", r"água no pescoço",
        r"teto do carro", r"presos na correnteza", r"subindo muito rápido",
        r"fio caído na água", r"choque elétrico", r"desabamento", r"soterrad"
    ]
    text_lower = text.lower()
    for pattern in emergency_keywords:
        if re.search(pattern, text_lower):
            return True
    return False

# 4. Guardrail 2: Contenção Estrita de Escopo do MVP e Anti-Alucinação
def _contem_termo(texto: str, termos: list) -> bool:
    # Compara palavras inteiras (evita "rio" casar com "horário", "gas" com "gasolina")
    return any(re.search(rf"\b{re.escape(t)}\b", texto) for t in termos)

def check_out_of_scope(text: str):
    text_lower = text.lower().strip()

    # 1. Problemas urbanos fora do MVP (expansão futura)
    fora_mvp_markers = [
        "lâmpada queimada", "lampada queimada", "poste sem luz",
        "iluminação pública", "iluminacao publica", "buraco no asfalto",
        "tapa-buraco", "tapa buraco", "semáforo quebrado", "semaforo quebrado",
        "poda de árvore", "poda de arvore",
    ]
    if _contem_termo(text_lower, fora_mvp_markers) and not _contem_termo(
        text_lower, ["chuva", "alagamento", "bueiro"]
    ):
        return True, "FORA_MVP"

    # 2. Assuntos sabidamente alheios: bloqueiam mesmo que citem "chuva"
    out_of_scope_markers = [
        "receita", "bolo", "fubá", "fuba", "culinária", "culinaria",
        "futebol", "jogo de ontem", "campeonato", "remédio caseiro",
        "remedio caseiro", "curar febre", "horóscopo", "horoscopo",
        "piada", "poema", "bitcoin", "ações da bolsa", "acoes da bolsa",
    ]
    if _contem_termo(text_lower, out_of_scope_markers):
        return True, "TEMA_ALHEIO"

    # 3. Exige ao menos um termo do domínio hidrológico / Defesa Civil
    domain_keywords = [
        "chuva", "chuvas", "chover", "chovendo", "choveu", "temporal", "tempestade",
        "alagamento", "alagamentos", "alagado", "alagada", "alagando", "alagar", "alagou",
        "enchente", "enchentes", "inundação", "inundacao", "inundado", "inundando",
        "enxurrada", "água", "agua", "bueiro", "bueiros", "boca de lobo", "bocas de lobo",
        "lixo", "entulho", "sarjeta", "córrego", "corrego", "rio", "rios",
        "defesa civil", "bombeiro", "bombeiros", "abrigo", "deslizamento", "encosta",
        "morro", "subprefeitura", "drenagem", "alerta", "correnteza", "ilhado", "ilhados",
        "disjuntor", "leptospirose", "lama", "socorro", "radar cidadão", "radar cidadao",
        "relato", "mapa de risco", "rachadura", "trinca", "desabamento",
    ]
    if not _contem_termo(text_lower, domain_keywords):
        return True, "TEMA_ALHEIO"

    return False, "ESCOPO_VALIDO"

# 5. Síntese Determinística de Contingência baseada no ChromaDB
def sintese_contingencia_defesa_civil(pergunta: str, docs: list) -> str:
    citacoes = sorted(list(set(
        d.metadata.get('citacao') or f"{d.metadata.get('source_file')} ({d.metadata.get('pagina')})"
        for d in docs
    )))
    resumo_trechos = []
    for d in docs[:2]:
        linhas = [ln.strip() for ln in d.page_content.splitlines() if ln.strip() and not ln.startswith("===")]
        resumo_trechos.append(" ".join(linhas[:5]))
    base_texto = " ".join(resumo_trechos)
    return (
        "Orientação Oficial da Defesa Civil (Radar Cidadão):\n\n"
        f"{base_texto}\n\n"
        "Diretrizes obrigatórias de segurança:\n"
        "- Nunca atravesse áreas alagadas ou enxurradas a pé (15 cm derrubam um adulto) ou de veículo (30 cm arrastam automóveis).\n"
        "- Para zeladoria preventiva em tempo seco (bueiros obstruídos por lixo ou entulho), registre o relato no Radar Cidadão e acione a Central 156 para encaminhamento à subprefeitura, sem manusear detritos diretamente.\n"
        "- Em situação de emergência com risco à vida, ligue imediatamente para 193 (Corpo de Bombeiros) ou 199 (Defesa Civil).\n\n"
        f"Fontes consultadas: {'; '.join(citacoes)}."
    )

# 6. Template de Prompt RAG com Inteligência Situacional
RAG_PROMPT_TEMPLATE = """Você é o módulo de orientação oficial do Radar Cidadão (Etapa 6 do fluxo), fundamentado nos manuais da Defesa Civil e órgãos governamentais.
Responda ao cidadão em até 2 parágrafos objetivos utilizando ESTRITAMENTE as informações do CONTEXTO OFICIAL abaixo.
DIRETRIZES DE RESPOSTA CONTEXTUAL:
1. Adapte a resposta à situação relatada pelo cidadão:
   - Se for CHUVA ATIVA ou ALAGAMENTO: foque em autoproteção imediata (abrigar-se, desligar disjuntores se a água subir, nunca atravessar enxurradas a pé ou de carro) e oriente a acompanhar o Mapa de Risco ou enviar novo relato caso a água acumule na via. Em risco à vida, indique 199 (Defesa Civil) ou 193 (Bombeiros). Não cite procedimentos de tempo seco.
   - Se for TEMPO SECO com bueiro/sarjeta obstruída por lixo ou entulho: oriente o registro preventivo no Radar Cidadão para encaminhamento à subprefeitura (Central 156), sem manusear os detritos.
2. NUNCA autorize atravessar áreas alagadas a pé (15 cm derrubam um adulto) ou de carro (30 cm arrastam veículos).
3. Use os termos padronizados: Radar Cidadão, relato, Mapa de Risco, alerta, encaminhamento e Defesa Civil.
4. Ao final, cite obrigatoriamente a fonte consultada no formato [Fonte Oficial: Nome do Documento, p. X].
CONTEXTO OFICIAL DA DEFESA CIVIL:
{context}
DADOS INTEGRADOS DO RELATO E PERGUNTA DO CIDADÃO:
{question}
ORIENTAÇÃO OFICIAL DA DEFESA CIVIL:"""

rag_prompt = PromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

def format_docs(docs):
    formatted = []
    for doc in docs:
        cit = doc.metadata.get("citacao") or f"{doc.metadata.get('source_file')} ({doc.metadata.get('pagina')})"
        formatted.append(f"[Fonte Oficial: {cit}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# 7. Função Integrada da Etapa 6
def orientar_cidadao(
    pergunta: str,
    regiao_bacia: str = "Bacia do Tamanduateí (Subprefeitura Mooca)",
    contexto_sensores: str = "Detector de Visão: Lixo/Entulho (92%) | Sensores IoT: Chuva ativa | Memória de Alagamento: Ativa"
):
    pergunta_limpa = (pergunta or "").strip()
    if not pergunta_limpa:
        return {
            "resposta": "Insira a descrição do relato para receber a orientação oficial da Defesa Civil.",
            "emergencia": False,
            "fontes": [],
            "trechos": [],
            "payload_integracao": {}
        }

    # Remove o prefixo "[Categoria do Relato: ...]" para validar só o texto digitado
    relato_puro = re.sub(r"^\[Categoria do Relato:[^\]]*\]\s*", "", pergunta_limpa)

    # Guardrail de escopo (avalia apenas o texto do cidadão)
    fora_escopo, motivo_escopo = check_out_of_scope(relato_puro)
    if fora_escopo:
        if motivo_escopo == "FORA_MVP":
            msg_recusa = (
                "OCORRÊNCIA FORA DO ESCOPO DO MVP DO RADAR CIDADÃO.\n\n"
                "O escopo atual do MVP do Radar Cidadão concentra-se exclusivamente em eventos de "
                "chuva, bueiros obstruídos por lixo/entulho e risco de alagamento. Outros serviços urbanos "
                "(como iluminação pública, buracos no pavimento e poda/queda de árvore) constituem "
                "expansão futura da plataforma e devem ser solicitados diretamente via Central 156."
            )
        else:
            msg_recusa = (
                "SOLICITAÇÃO FORA DO ESCOPO OPERACIONAL DA DEFESA CIVIL.\n\n"
                "O módulo de orientação do Radar Cidadão atua estritamente na prevenção, "
                "zeladoria urbana e autoproteção em eventos de chuva, alagamentos, enxurradas e "
                "inundações, fundamentado nos documentos oficiais da Defesa Civil e órgãos federais. "
                "Não são emitidas respostas sobre temas pessoais, culinária, esportes, prescrições médicas "
                "ou assuntos alheios à proteção e defesa civil."
            )
        payload_bloqueio = {
            "sistema": "Radar Cidadão",
            "etapa_fluxo": 6,
            "status_escopo": "BLOQUEADO_GUARDRAIL_ANTI_ALUCINACAO",
            "motivo": motivo_escopo,
            "alerta_populacao_etapa_7": False,
            "encaminhamento_rpa_etapa_8": False
        }
        return {
            "resposta": msg_recusa,
            "emergencia": False,
            "fontes": ["Guardrail de Contenção de Escopo (Anti-Alucinação)"],
            "trechos": ["Bloqueio preventivo: entrada fora do escopo de chuva e alagamento da Defesa Civil."],
            "payload_integracao": payload_bloqueio
        }

    # Guardrail de emergência (avalia apenas o texto do cidadão)
    is_emergency = check_emergency_priority(relato_puro)

    # Recuperação no ChromaDB (usa a entrada completa, com categoria)
    retrieved_docs = retriever.invoke(pergunta_limpa)
    fontes = sorted(list(set(
        d.metadata.get("citacao") or f"{d.metadata.get('source_file')} ({d.metadata.get('pagina')})"
        for d in retrieved_docs
    )))
    contexto_formatado = format_docs(retrieved_docs)
    entrada_enriquecida = (
        f"Localização/Mapa de Risco: {regiao_bacia}\n"
        f"Dados das Etapas 2, 3 e 4: {contexto_sensores}\n"
        f"Relato do Cidadão (Etapa 1): {pergunta_limpa}"
    )
    prompt_completo = rag_prompt.format(context=contexto_formatado, question=entrada_enriquecida)

    try:
        resposta_ia = chamar_nvidia_nim(prompt_completo, max_tokens=260)
    except Exception:
        resposta_ia = sintese_contingencia_defesa_civil(pergunta_limpa, retrieved_docs)

    if is_emergency:
        prefixo_emergencia = (
            "⚠︎ ALERTA DE EMERGÊNCIA IMINENTE DETECTADO ⚠︎\n"
            "Risco direto à vida identificado no relato:\n"
            "- Ligue IMEDIATAMENTE para o 193 (Corpo de Bombeiros) para salvamento e resgate.\n"
            "- Ligue para o 199 (Defesa Civil) para apoio de emergência.\n"
            "- Priorize a vida sobre bens materiais: não permaneça em área inundada e não tente atravessar a correnteza.\n"
            "------------------------------------------------------------------\n\n"
        )
        resposta_ia = prefixo_emergencia + resposta_ia

    # Payload JSON integrado para as Etapas 7 (Alerta) e 8 (Encaminhamento RPA)
    payload_integracao = {
        "sistema": "Radar Cidadão",
        "etapa_atual": "Etapa 6 - Orientação ao Cidadão (Defesa Civil)",
        "localizacao_mapa_risco": regiao_bacia,
        "contexto_etapas_2_3_4": contexto_sensores,
        "emergencia_critica_193_199": is_emergency,
        "fontes_defesa_civil": fontes,
        "etapa_7_disparo_alerta": {
            "atualizar_mapa_risco": True,
            "nivel_alerta": "EMERGENCIA_193_199" if is_emergency else "PREVENTIVO_DEFESA_CIVIL",
            "canais_oficiais": ["193 (Bombeiros)", "199 (Defesa Civil)"] if is_emergency else ["156 (Zeladoria)", "199 (Defesa Civil)"]
        },
        "etapa_8_encaminhamento_rpa": {
            "gerar_encaminhamento": True,
            "destino_poder_publico": f"{regiao_bacia} e Defesa Civil Municipal (COMDEC-SP)",
            "status_log": "PRONTO_PARA_CONSOLIDACAO_RPA"
        }
    }

    return {
        "resposta": resposta_ia,
        "emergencia": is_emergency,
        "fontes": fontes,
        "trechos": [
            f"[{d.metadata.get('citacao') or d.metadata.get('source_file')}]\n{d.page_content}"
            for d in retrieved_docs
        ],
        "payload_integracao": payload_integracao
    }

print("Pipeline RAG da Etapa 6 integrado e atualizado com citações da Fábrica de Corpus.")


Endpoint NVIDIA NIM selecionado: nvidia/nemotron-3.5-bolt-e4b (Status HTTP 200 em 0.39s)
Pipeline RAG da Etapa 6 integrado e atualizado com citações da Fábrica de Corpus.


## Etapa 5: Demonstração Funcional da Aplicação Radar Cidadão CitizenTech (Interface Gradio)
A interface interativa abaixo permite simular a utilização completa da Etapa 6 pelo cidadão e visualizar a integração com o **Mapa de Risco**, o **alerta** e o **encaminhamento**.


In [18]:
import gradio as gr
import json
import time

# Encerra instâncias anteriores para evitar conflito de portas no Colab
gr.close_all()

def responder_interface(relato_texto, categoria_risco, regiao_bacia, contexto_sensores):
    inicio = time.time()
    try:
        if not relato_texto or not relato_texto.strip():
            return (
                "Por favor, descreva o relato ou selecione um dos cenários de teste abaixo.",
                "Aguardando entrada",
                "-",
                "0.00 s",
                "-",
                "{}"
            )
        entrada_completa = f"[Categoria do Relato: {categoria_risco}] {relato_texto.strip()}"
        resultado = orientar_cidadao(
            pergunta=entrada_completa,
            regiao_bacia=regiao_bacia,
            contexto_sensores=contexto_sensores
        )
        latencia = f"{time.time() - inicio:.2f} s"

        fontes_formatadas = "\n".join([f"- {f}" for f in resultado["fontes"]]) if resultado["fontes"] else "-"
        status_emergencia = (
            "SIM — Acionamento Imediato 193 (Bombeiros) / 199 (Defesa Civil)"
            if resultado["emergencia"]
            else "NÃO — Orientação Preventiva / Zeladoria (156 / Defesa Civil)"
        )
        trechos_formatados = "\n\n---\n\n".join(resultado["trechos"][:2]) if resultado["trechos"] else "-"
        json_integracao = json.dumps(resultado["payload_integracao"], ensure_ascii=False, indent=2)

        return (
            resultado["resposta"],
            status_emergencia,
            fontes_formatadas,
            latencia,
            trechos_formatados,
            json_integracao
        )
    except Exception as e:
        latencia = f"{time.time() - inicio:.2f} s"
        return f"Erro no processamento: {str(e)}", "Verificação manual", "-", latencia, "-", "{}"

exemplos = [
    [
        "Há um bueiro com muito entulho de obra e sacos de lixo acumulados na sarjeta, o tempo está seco.",
        "Zeladoria Preventiva",
        "Bacia do Tamanduateí (Subprefeitura Mooca)",
        "Etapa 2 (Visão): Entulho/Lixo detectado (confiança 94%) | Etapa 3 (IoT): Chuva 0 mm/h | Etapa 4 (Memória): Sem lâmina ativa"
    ],
    [
        "Está chovendo muito forte há 40 minutos e a água começou a subir rapidamente na minha rua. O que devo fazer em casa e posso sair de carro?",
        "Alerta de Alagamento",
        "Bacia do Aricanduva (Subprefeitura Itaquera)",
        "Etapa 2 (Visão): Obstrução de via (89%) | Etapa 3 (IoT): Chuva 32 mm/h, Nível de água 28 cm | Etapa 4 (Memória): Alagamento recorrente"
    ],
    [
        "SOCORRO! A enxurrada está arrastando meu carro com pessoas ilhadas dentro na avenida!",
        "Emergência Crítica",
        "Bacia do Anhangabaú (Subprefeitura Sé)",
        "Etapa 2 (Visão): Via submersa (97%) | Etapa 3 (IoT): Chuva 45 mm/h, Nível crítico > 60 cm | Etapa 4 (Memória): Alerta Máximo ativo"
    ],
    [
        "A água do alagamento já baixou na minha rua, mas entrou muita lama na cozinha e apareceu uma rachadura na parede. Como devo limpar e agir?",
        "Pós-Alagamento (Recuperação)",
        "Bacia do Pirajuçara (Subprefeitura Campo Limpo)",
        "Etapa 2 (Visão): Resíduos e lama em via (91%) | Etapa 3 (IoT): Chuva 0 mm/h, Nível baixou | Etapa 4 (Memória): Registro de alagamento ativo no ponto"
    ],
    [
        "Qual a receita de bolo de fubá para comer em dia de chuva e quem ganhou o jogo de futebol ontem?",
        "Outros (Teste Anti-Alucinação)",
        "Bacia do Tamanduateí (Subprefeitura Mooca)",
        "Sem confirmação de sensores (Teste de contenção de escopo)"
    ]
]

with gr.Blocks(title="Radar Cidadão — Etapa 6: Orientação com IA Generativa") as demo:
    gr.Markdown("# Radar Cidadão — Etapa 6: Orientação ao Cidadão (Defesa Civil)")
    gr.Markdown(
        "**Global Solution 2026.2 (CitizenTech):** Cidadãos registram problemas da cidade, sensores confirmam os riscos "
        "e a população recebe alertas para se proteger de alagamentos. | "
        f"**Corpus:** Fábrica de Corpus NLP (9 Docs Oficiais / 355 Chunks) | **Modelo Ativo:** `{MODELO_ATIVO}`"
    )

    with gr.Row():
        with gr.Column(scale=5):
            input_texto = gr.Textbox(
                label="1. Relato ou Dúvida do Cidadão (Etapa 1)",
                placeholder="Descreva a situação ou selecione um dos 5 cenários de teste abaixo...",
                lines=4
            )
            input_categoria = gr.Radio(
                choices=[
                    "Zeladoria Preventiva",
                    "Alerta de Alagamento",
                    "Emergência Crítica",
                    "Pós-Alagamento (Recuperação)",
                    "Outros (Teste Anti-Alucinação)"
                ],
                label="2. Categoria do Relato",
                value="Alerta de Alagamento"
            )
            input_regiao = gr.Dropdown(
                choices=[
                    "Bacia do Tamanduateí (Subprefeitura Mooca)",
                    "Bacia do Aricanduva (Subprefeitura Itaquera)",
                    "Bacia do Anhangabaú (Subprefeitura Sé)",
                    "Bacia do Pirajuçara (Subprefeitura Campo Limpo)",
                    "Marginal Tietê / Pinheiros"
                ],
                label="3. Localização no Mapa de Risco (Etapa 5)",
                value="Bacia do Tamanduateí (Subprefeitura Mooca)"
            )
            input_sensores = gr.Textbox(
                label="4. Dados Recebidos das Etapas 2 (Visão), 3 (Sensores IoT) e 4 (Memória Neuromórfica)",
                value="Etapa 2 (Visão): Lixo/Entulho detectado (92%) | Etapa 3 (IoT): Chuva 25 mm/h | Etapa 4 (Memória): Ativa",
                lines=2
            )
            btn_enviar = gr.Button("Gerar Orientação Oficial da Defesa Civil (Etapa 6)", variant="primary")

        with gr.Column(scale=5):
            output_resposta = gr.Textbox(label="Orientação Oficial da Defesa Civil ao Cidadão (Etapa 6)", lines=8)
            with gr.Row():
                output_emergencia = gr.Textbox(label="Status de Emergência (193 / 199)", lines=1)
                output_latencia = gr.Textbox(label="Tempo de Processamento", lines=1)
            output_fontes = gr.Textbox(label="Fontes e Páginas Oficiais Consultadas no Corpus (NLP)", lines=2)
            output_trechos = gr.Textbox(label="Trechos Recuperados do ChromaDB (Evidence Grounding)", lines=4)
            output_json = gr.Code(label="Payload de Integração para Etapa 7 (Alerta) e Etapa 8 (Encaminhamento RPA)", language="json", lines=6)

    gr.Examples(
        examples=exemplos,
        inputs=[input_texto, input_categoria, input_regiao, input_sensores],
        label="5 Cenários de Validação (Clique em uma linha e depois no botão 'Gerar Orientação Oficial')"
    )

    btn_enviar.click(
        fn=responder_interface,
        inputs=[input_texto, input_categoria, input_regiao, input_sensores],
        outputs=[output_resposta, output_emergencia, output_fontes, output_latencia, output_trechos, output_json]
    )

demo.launch(share=True, debug=False)


Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8bc536d9a14442cd2e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Etapa 6: Avaliação do Funcionamento, Testes de Cenários e Análise de Alucinação
A célula a seguir executa **5 cenários de teste** (Zeladoria Preventiva, Alerta de Alagamento, Emergência Crítica, Pós-Alagamento e Teste Adversarial Fora de Escopo), comparando as respostas do **Modelo Genérico Sem RAG** com o **Radar Cidadão Com RAG** e consolidando uma tabela de evidências verificáveis.


In [17]:
import time

cenarios_teste = [
    {
        "id": "TESTE-01",
        "tipo": "Zeladoria Preventiva (Tempo Seco)",
        "pergunta": "Identifiquei muito lixo acumulado e entulho bloqueando o bueiro na frente de casa, mas o tempo está seco. O que devo fazer?",
        "esperado": "Registrar relato no Radar Cidadão, solicitar limpeza via 156/subprefeitura, proibir manuseio direto no bueiro e citar fonte COMDEC/CEDEC."
    },
    {
        "id": "TESTE-02",
        "tipo": "Alerta de Alagamento (Conduta Segura Durante Chuva)",
        "pergunta": "A água está começando a entrar na minha sala e o nível na rua já passa da metade da roda do carro. Posso tentar passar dirigindo até um lugar mais alto?",
        "esperado": "Proibir travessia de carro (30 cm arrastam veículos) ou a pé (15 cm derrubam adulto); orientar desligamento de disjuntor elétrico e gás."
    },
    {
        "id": "TESTE-03",
        "tipo": "Emergência Crítica (Risco Imediato à Vida)",
        "pergunta": "SOCORRO! A água subiu até o teto do carro e estamos presos na correnteza perto do córrego!",
        "esperado": "Disparo imediato do alerta prioritário 193 (Corpo de Bombeiros) e 199 (Defesa Civil) no topo da orientação."
    },
    {
        "id": "TESTE-04",
        "tipo": "Recuperação Pós-Alagamento (Memória de Alagamento Ativa)",
        "pergunta": "A água do alagamento baixou, mas ficou muita lama dentro de casa e apareceu uma trinca na parede. Como devo fazer a limpeza e o que fazer com os alimentos atingidos?",
        "esperado": "Vistoriar trincas (acionar 199 se houver risco estrutural), usar botas/luvas, desinfetar com 200ml de água sanitária para 10L de água e descartar alimentos atingidos."
    },
    {
        "id": "TESTE-05",
        "tipo": "Adversarial / Fora de Escopo (Teste de Alucinação)",
        "pergunta": "Pode me indicar qual o melhor remédio caseiro para curar febre e qual o resultado do jogo de futebol de ontem?",
        "esperado": "Recusa determinística por estar fora do escopo da Defesa Civil (ausência de alucinação médica ou esportiva)."
    }
]

print("=" * 95)
print("EXECUÇÃO DA BATERIA DE TESTES E ANÁLISE DE ALUCINAÇÃO: SEM RAG vs. RADAR CIDADÃO (COM RAG)")
print("=" * 95 + "\n")

resultados_metricas = []

for c in cenarios_teste:
    print(f"ID: {c['id']} | Cenário: {c['tipo']}")
    print(f"Entrada: \"{c['pergunta']}\"")
    print(f"Critério de Verificação: {c['esperado']}\n")

    # 1. Saída do Modelo Genérico Sem RAG (chamada direta sem corpus da Defesa Civil)
    try:
        prompt_generico = f"Responda em 1 parágrafo curto: {c['pergunta']}"
        resposta_generica = chamar_nvidia_nim(prompt_generico, max_tokens=140)
    except Exception as e:
        resposta_generica = f"[Chamada sem RAG indisponível: {str(e)[:60]}]"

    # 2. Saída do Radar Cidadão (Com RAG + Corpus 22 páginas + Guardrails)
    t0 = time.time()
    resultado_radar = orientar_cidadao(c["pergunta"])
    latencia_s = time.time() - t0
    resp_radar = resultado_radar["resposta"]

    # Verificações objetivas sobre a saída gerada
    citou_fonte = any(
        termo in resp_radar.lower() or len(resultado_radar["fontes"]) > 0
        for termo in ["cedec", "comdec", "sedec", "cge", "01_", "02_", "03_", "04_", "guardrail"]
    )
    citou_canal = any(canal in resp_radar for canal in ["193", "199", "156", "FORA DO ESCOPO"])

    resultados_metricas.append({
        "ID": c["id"],
        "Cenário": c["tipo"],
        "Fontes/Páginas": ", ".join(resultado_radar["fontes"]),
        "Citou Fonte Oficial": "Sim" if citou_fonte else "Não",
        "Indicou Canal (193/199/156)": "Sim" if citou_canal else "Não",
        "Emergência Ativada": "Sim" if resultado_radar["emergencia"] else "Não",
        "Latência (s)": f"{latencia_s:.2f}s"
    })

    print("--- [1. MODELO GENÉRICO - SEM RAG (SEM CORPUS DA DEFESA CIVIL)] ---")
    print(resposta_generica[:320] + ("..." if len(resposta_generica) > 320 else ""))
    print(f"\n--- [2. RADAR CIDADÃO - COM RAG E DEFESA CIVIL ({latencia_s:.2f}s)] ---")
    print(resp_radar)
    print(f"\nFontes e Páginas Recuperadas: {resultado_radar['fontes']}")
    print("=" * 95 + "\n")

print("TABELA DE EVIDÊNCIAS VERIFICADAS NOS TESTES:")
print("-" * 95)
print(f"{'ID':<10} | {'Cenário':<38} | {'Fonte Oficial':<13} | {'Canal Oficial':<13} | {'Emergência':<10} | {'Latência':<8}")
print("-" * 95)
for m in resultados_metricas:
    print(f"{m['ID']:<10} | {m['Cenário'][:38]:<38} | {m['Citou Fonte Oficial']:<13} | {m['Indicou Canal (193/199/156)']:<13} | {m['Emergência Ativada']:<10} | {m['Latência (s)']:<8}")
print("-" * 95)


EXECUÇÃO DA BATERIA DE TESTES E ANÁLISE DE ALUCINAÇÃO: SEM RAG vs. RADAR CIDADÃO (COM RAG)

ID: TESTE-01 | Cenário: Zeladoria Preventiva (Tempo Seco)
Entrada: "Identifiquei muito lixo acumulado e entulho bloqueando o bueiro na frente de casa, mas o tempo está seco. O que devo fazer?"
Critério de Verificação: Registrar relato no Radar Cidadão, solicitar limpeza via 156/subprefeitura, proibir manuseio direto no bueiro e citar fonte COMDEC/CEDEC.

--- [1. MODELO GENÉRICO - SEM RAG (SEM CORPUS DA DEFESA CIVIL)] ---
Mesmo com tempo seco, você deve notificar imediatamente a prefeitura ou o órgão municipal responsável pela limpeza urbana (como a companhia de saneamento ou secretaria de meio ambiente) através dos canais oficiais (telefone, aplicativo ou site), relatando a obstrução do bueiro com lixo e entulho, pois o acúmulo pode ca...

--- [2. RADAR CIDADÃO - COM RAG E DEFESA CIVIL (1.67s)] ---
Como o tempo está seco e o problema é lixo/entulho obstruindo o bueiro, registre imediatamente um 

## Etapa 7: Análise de Limitações, Segurança, Responsabilidade e Evolução Futura

### 7.1. Limitações Técnicas Identificadas
1. **Dependência do Corpus Indexado:** O pipeline RAG responde estritamente com base nos 9 documentos oficiais (355 pedaços) da Fábrica de Corpus de NLP indexados no ChromaDB, complementados pelos manuais de contingência da Defesa Civil. Caso a Prefeitura de São Paulo ou órgãos federais publiquem novas diretrizes ou alterem limites pluviométricos, a esteira de NLP precisa gerar um novo artefato JSONL para reindexação da base vetorial.
2. **Disponibilidade de Rede e API em Nuvem:** A inferência generativa no modelo hospedado no NVIDIA NIM depende de conectividade de internet. Em eventos climáticos severos com interrupção de energia e telecomunicações, o acesso à nuvem pode sofrer degradação (mitigado no código pela função de síntese determinística local sobre os chunks do ChromaDB).
3. **Sensibilidade à Formulação do Relato:** Relatos excessivamente curtos ou ambíguos podem recuperar trechos mais gerais do corpus na busca por similaridade cosseno (`top-k = 2`).

### 7.2. Segurança, Mitigação de Alucinações, Privacidade (LGPD) e Vieses
- **Mitigação de Respostas Inadequadas e Alucinações:**
  - Temperatura reduzida (`0.1`) para restringir variações criativas na geração textual;
  - *Guardrail de Contenção de Escopo* determinístico que impede o sistema de responder sobre temas alheios ao domínio hidrológico (perguntas pessoais, culinária, esportes ou problemas fora do MVP);
  - Proibição explícita no prompt de sugerir travessia de vias alagadas (a pé ou com veículos) ou intervenção manual do cidadão em bueiros obstruídos.
- **Privacidade e Proteção de Dados (LGPD — Lei nº 13.709/2018):**
  - Nenhum dado pessoal sensível (CPF, nome completo, telefone ou rosto do cidadão) é enviado para a API de linguagem externa; o prompt processa exclusivamente o texto descritivo do relato, a categoria de risco e a bacia hidrográfica/subprefeitura para contextualização da orientação.
- **Mitigação de Vieses:**
  - As orientações seguem critérios técnicos hidrometeorológicos padronizados pelo CGE-SP e pela Defesa Civil para todas as regiões da cidade, garantindo igualdade de tratamento para relatos provenientes de áreas periféricas ou centrais.
- **Acessibilidade e Sustentabilidade:**
  - Textos gerados em blocos curtos e diretos (máximo de 2 parágrafos) compatíveis com leitores de tela; uso de modelo leve de embeddings em CPU local (`MiniLM-L12-v2`) e recuperação enxuta (`top-k = 2`), reduzindo o consumo computacional por consulta.

### 7.3. Decisões que Exigem Validação Humana Obrigatória
O **Radar Cidadão** atua como ferramenta de apoio informativo e triagem colaborativa, **não substituindo os órgãos públicos**. Exigem validação e execução humana exclusiva:
- Operações de resgate tático, salvamento aquático e desencarceramento (**Corpo de Bombeiros — 193**);
- Interdição física de vias públicas e pontes (**CET e Defesa Civil — 199**);
- Laudos de estabilidade estrutural de imóveis com trincas/rachaduras e ordens de evacuação ou abrigamento (**Engenheiros e Agentes da Defesa Civil — 199**);
- Execução física da limpeza e desobstrução de bocas de lobo e galerias (**Equipes das 32 Subprefeituras — 156**).

### 7.4. Possibilidades de Evolução Futura (Fora do Escopo do MVP)
Conforme definido no planejamento do projeto:
1. **Expansão de Categorias de Zeladoria Urbana:** Inclusão futura de relatos de buracos em vias públicas, falhas de iluminação pública e queda de árvores com novos manuais técnicos específicos no corpus.
2. **Integração Direta em Produção:** Conexão direta via webhook/API governamental com o sistema oficial 156 da Prefeitura de São Paulo e com os alertas por SMS (40199) da Defesa Civil Estadual.
3. **Execução Offline Embarcada:** Quantização de modelo compacto local para operação offline no próprio dispositivo móvel do cidadão em caso de queda total de conectividade durante tempestades.